<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/Drake/OOPv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

class BodyConfig:
    """
    Configuration for a rigid body to simulate.
    """
    def __init__(self,
                 mass: float = 1.0,
                 size: list[float] = [0.2, 0.2, 0.2],
                 initial_position: list[float] = [0, 0, 1.0],
                 gravity: list[float] = [0, 0, -9.81],
                 friction: tuple[float,float] = (0.9, 0.8),
                 inertia_matrix: np.ndarray = None):
        """
        mass: scalar mass in kg
        size: [x, y, z] dimensions in meters (for box placeholder)
        initial_position: [x, y, z] start location
        gravity: gravity vector
        friction: (static, dynamic)
        inertia_matrix: optional 3x3 numpy array, custom inertia tensor
        """
        self.mass = mass
        self.size = size
        self.initial_position = initial_position
        self.gravity = gravity
        self.friction = friction
        self.inertia_matrix = inertia_matrix

In [ ]:
from pydrake.all import *

class RigidBodySimulator:
    """
    Wrapper around Drake to simulate a single rigid body
    with customizable physics.
    """
    def __init__(self, config: BodyConfig, time_step: float = 0.001):
        self.config = config
        self.time_step = time_step
        self._build_system()

    def _build_system(self):
        # 1. Diagram & plant
        self.builder = DiagramBuilder()
        self.plant, self.scene_graph = AddMultibodyPlantSceneGraph(
            self.builder,
            MultibodyPlant(time_step=self.time_step)
        )

        # 2. Add body
        self._add_body()

        # 3. Add ground
        self._add_ground()

        # 4. Finalize plant
        self.plant.Finalize()

        # 5. Meshcat visualizer
        self.meshcat = StartMeshcat()
        MeshcatVisualizer.AddToBuilder(
            self.builder,
            self.scene_graph,
            self.meshcat
        )

        # 6. Build diagram
        self.diagram = self.builder.Build()
        self.simulator = Simulator(self.diagram)

    def _add_body(self):
        cfg = self.config

        # Use custom inertia if provided, else solid box
        if cfg.inertia_matrix is None:
            inertia = UnitInertia.SolidBox(*cfg.size)
        else:
            I = cfg.inertia_matrix
            inertia = UnitInertia(
                Ixx=I[0,0], Iyy=I[1,1], Izz=I[2,2],
                Ixy=I[0,1], Ixz=I[0,2], Iyz=I[1,2]
            )

        spatial_inertia = SpatialInertia(
            mass=cfg.mass,
            p_PScm_E=np.zeros(3),
            G_SP_E=inertia
        )

        # Add the rigid body
        self.body = self.plant.AddRigidBody("body", spatial_inertia)

        # Set initial pose
        self.plant.SetDefaultFloatingBaseBodyPose(
            self.body,
            RigidTransform(cfg.initial_position)
        )

        # Visual geometry placeholder
        shape = Box(*cfg.size)
        self.plant.RegisterVisualGeometry(
            self.body,
            RigidTransform(),
            shape,
            "body_visual",
            [0.5, 0.5, 1.0, 1.0]
        )

        # Collision geometry
        self.plant.RegisterCollisionGeometry(
            self.body,
            RigidTransform(),
            shape,
            "body_collision",
            CoulombFriction(*cfg.friction)
        )

        # Gravity
        self.plant.mutable_gravity_field().set_gravity_vector(cfg.gravity)

    def _add_ground(self):
        ground_shape = HalfSpace()
        X_WG = RigidTransform(RollPitchYaw(np.pi, 0, 0), [0, 0, 0])

        self.plant.RegisterCollisionGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_collision",
            CoulombFriction(0.9, 0.8)
        )

        self.plant.RegisterVisualGeometry(
            self.plant.world_body(),
            X_WG,
            ground_shape,
            "ground_visual",
            [0.5, 0.5, 0.5, 1.0]
        )

    def simulate(self, duration: float = 5.0, realtime_rate: float = 1.0):
        """
        Run the simulation.
        """
        self.simulator.set_target_realtime_rate(realtime_rate)
        self.simulator.Initialize()
        self.simulator.AdvanceTo(duration)

    def get_web_url(self) -> str:
        """
        Returns the Meshcat URL to view the simulation.
        """
        return self.meshcat.web_url()

    def get_state(self):
        """
        Returns current positions (q) and velocities (v)
        """
        context = self.simulator.get_context()
        plant_context = self.plant.GetMyContextFromRoot(context)
        q = self.plant.GetPositions(plant_context)
        v = self.plant.GetVelocities(plant_context)
        return q, v
